# 🛍️ Product Discovery using Fashion-MNIST
> **Neural Networks & Deep Learning — Mini Project**  
> Academic Grade Target: **9.5 / 10**

---
## 📌 Problem Statement
Online marketplaces receive thousands of new clothing items daily. Manual product categorisation is slow and error-prone.  
This notebook demonstrates how a deep learning pipeline can automatically classify clothing images into **10 categories** using the Fashion-MNIST dataset.

## 🎯 Objectives
1. Load and explore the Fashion-MNIST dataset
2. Build a baseline **ANN** (Fully-Connected Network)
3. Build an improved **CNN** (Convolutional Neural Network)
4. Compare both models on accuracy, loss, confusion matrix, and classification report
5. Visualise predictions and errors
6. Discuss real-world applications

---
## 📦 Step 0 — Imports & Setup

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers, callbacks
from sklearn.metrics import confusion_matrix, classification_report

# Add project root to path so we can import src modules
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# Reproducibility
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow version : {tf.__version__}')
print(f'NumPy version      : {np.__version__}')
print(f'GPU available      : {len(tf.config.list_physical_devices("GPU")) > 0}')

---
## 📊 Step 1 — Dataset Loading & Exploration

In [ ]:
CLASS_NAMES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]
NUM_CLASSES = len(CLASS_NAMES)

# Load dataset
(X_train_raw, y_train), (X_test_raw, y_test) = keras.datasets.fashion_mnist.load_data()

print('Dataset Summary')
print('=' * 40)
print(f'Train samples : {X_train_raw.shape[0]:,}')
print(f'Test  samples : {X_test_raw.shape[0]:,}')
print(f'Image shape   : {X_train_raw.shape[1:]} (H x W)')
print(f'Pixel range   : [{X_train_raw.min()}, {X_train_raw.max()}]')
print(f'Classes       : {NUM_CLASSES}')

In [ ]:
# ── Visualise sample images ──────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 6))
fig.patch.set_facecolor('#1a1a2e')
gs  = gridspec.GridSpec(4, 10, figure=fig, hspace=0.5, wspace=0.3)

for cls in range(NUM_CLASSES):
    idxs = np.where(y_train == cls)[0][:4]
    for row, idx in enumerate(idxs):
        ax = fig.add_subplot(gs[row, cls])
        ax.imshow(X_train_raw[idx], cmap='gray')
        if row == 0:
            ax.set_title(CLASS_NAMES[cls], fontsize=6.5, color='white', pad=3)
        ax.axis('off')

fig.suptitle('Fashion-MNIST — 4 Examples per Class', fontsize=14,
             color='white', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/plots/nb_sample_images.png',
            bbox_inches='tight', dpi=150, facecolor=fig.get_facecolor())
plt.show()

In [ ]:
# ── Class distribution ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.patch.set_facecolor('#1a1a2e')
palette = plt.cm.plasma(np.linspace(0.2, 0.9, NUM_CLASSES))

for ax, (y, split) in zip(axes, [(y_train, 'Train'), (y_test, 'Test')]):
    counts = [np.sum(y == i) for i in range(NUM_CLASSES)]
    bars   = ax.bar(CLASS_NAMES, counts, color=palette, edgecolor='white', lw=0.5)
    ax.set_facecolor('#16213e')
    ax.set_title(f'{split} Set Class Distribution', color='white',
                 fontsize=12, fontweight='bold')
    ax.tick_params(colors='white', labelrotation=45)
    ax.set_ylabel('Count', color='white')
    for spine in ax.spines.values(): spine.set_edgecolor('#444')
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{cnt:,}', ha='center', va='bottom', color='white', fontsize=8)

fig.suptitle('Balanced Dataset — Each Class ~6,000 Train / ~1,000 Test',
             color='white', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/plots/nb_class_distribution.png',
            bbox_inches='tight', dpi=150, facecolor=fig.get_facecolor())
plt.show()

print('\n✅ Dataset is balanced — no class-imbalance problem!')

---
## ⚙️ Step 2 — Preprocessing

In [ ]:
# Normalize pixel values to [0, 1]
X_train = X_train_raw.astype('float32') / 255.0
X_test  = X_test_raw.astype('float32')  / 255.0

# ANN: flatten to (N, 784)
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat  = X_test.reshape(X_test.shape[0],  -1)

# CNN: add channel dim → (N, 28, 28, 1)
X_train_4d = X_train[..., np.newaxis]
X_test_4d  = X_test[...,  np.newaxis]

print('Preprocessing complete!')
print(f'  ANN input shape  : {X_train_flat.shape}')
print(f'  CNN input shape  : {X_train_4d.shape}')
print(f'  Pixel range      : [{X_train.min():.1f}, {X_train.max():.1f}]')

---
## 🧠 Step 3 — Model 1: ANN (Fully-Connected Network)

**Architecture:**  
```
Input(784) → Dense(512,ReLU)+BN+Dropout
           → Dense(256,ReLU)+BN+Dropout
           → Dense(128,ReLU)+BN+Dropout
           → Dense(10, Softmax)
```

| Feature | Detail |
|---------|--------|
| Layers | 3 hidden Dense layers |
| Activations | ReLU (hidden), Softmax (output) |
| Regularization | L2 + BatchNorm + Dropout(0.3) |
| Optimizer | Adam(lr=0.001) |
| Loss | Sparse Categorical Crossentropy |

In [ ]:
def build_ann(input_dim=784, hidden_units=[512, 256, 128],
              dropout_rate=0.3, learning_rate=1e-3):
    """Build and compile a fully-connected ANN classifier."""
    model = models.Sequential(name='ANN_FashionMNIST')
    model.add(layers.InputLayer(input_shape=(input_dim,)))

    for i, units in enumerate(hidden_units):
        model.add(layers.Dense(units, activation='relu',
                               kernel_regularizer=regularizers.l2(1e-4),
                               name=f'dense_{i+1}'))
        model.add(layers.BatchNormalization(name=f'bn_{i+1}'))
        model.add(layers.Dropout(dropout_rate, name=f'dropout_{i+1}'))

    model.add(layers.Dense(NUM_CLASSES, activation='softmax', name='output'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

ann = build_ann()
ann.summary()

In [ ]:
# ── Callbacks ────────────────────────────────────────────────────────────────
def get_callbacks(model_name, patience=5):
    return [
        callbacks.EarlyStopping(monitor='val_accuracy', patience=patience,
                                restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                    patience=3, min_lr=1e-6, verbose=1),
        callbacks.ModelCheckpoint(f'../models/{model_name}_best.keras',
                                  monitor='val_accuracy', save_best_only=True)
    ]

# ── Train ANN ─────────────────────────────────────────────────────────────────
print('Training ANN...')
history_ann = ann.fit(
    X_train_flat, y_train,
    epochs=30,
    batch_size=64,
    validation_split=0.1,
    callbacks=get_callbacks('ANN'),
    verbose=1
)

In [ ]:
# ── Plot ANN Training Curves ──────────────────────────────────────────────────
def plot_curves(history, model_name):
    h = history.history
    epochs = range(1, len(h['accuracy']) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('#1a1a2e')
    for ax in (ax1, ax2):
        ax.set_facecolor('#16213e')
        ax.tick_params(colors='white')
        ax.yaxis.label.set_color('white')
        ax.xaxis.label.set_color('white')
        for s in ax.spines.values(): s.set_edgecolor('#444')

    ax1.plot(epochs, h['accuracy'],     color='#00d2ff', lw=2, label='Train')
    ax1.plot(epochs, h['val_accuracy'], color='#ff6b6b', lw=2, ls='--', label='Validation')
    ax1.set_title(f'{model_name} — Accuracy', color='white', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
    ax1.legend(facecolor='#1a1a2e', edgecolor='#555')
    ax1.grid(True, ls='--', alpha=0.4)

    ax2.plot(epochs, h['loss'],     color='#00d2ff', lw=2, label='Train')
    ax2.plot(epochs, h['val_loss'], color='#ff6b6b', lw=2, ls='--', label='Validation')
    ax2.set_title(f'{model_name} — Loss', color='white', fontsize=13, fontweight='bold')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
    ax2.legend(facecolor='#1a1a2e', edgecolor='#555')
    ax2.grid(True, ls='--', alpha=0.4)

    plt.tight_layout()
    plt.savefig(f'../outputs/plots/nb_{model_name}_curves.png',
                bbox_inches='tight', dpi=150, facecolor=fig.get_facecolor())
    plt.show()
    print(f'  Best val_accuracy : {max(h["val_accuracy"]):.4f}')
    print(f'  Epochs run        : {len(h["loss"])}')

plot_curves(history_ann, 'ANN')

In [ ]:
# ── Evaluate ANN ──────────────────────────────────────────────────────────────
ann_loss, ann_acc = ann.evaluate(X_test_flat, y_test, verbose=0)
print(f'ANN Test Accuracy : {ann_acc*100:.2f}%')
print(f'ANN Test Loss     : {ann_loss:.4f}')

---
## 🔬 Step 4 — Model 2: CNN (Convolutional Neural Network)

**Architecture:**
```
Input(28,28,1)
  → [Data Augmentation]
  → [Conv32→Conv32→MaxPool→Dropout]
  → [Conv64→Conv64→MaxPool→Dropout]
  → [Conv128→MaxPool→Dropout]
  → Flatten
  → Dense(256)+BN+Dropout
  → Dense(128)+BN+Dropout
  → Dense(10, Softmax)
```

| Feature | Detail |
|---------|--------|
| Conv filters | 32 → 64 → 128 |
| Pooling | MaxPool (2×2) |
| Augmentation | Rotation±10°, Shift±10%, Zoom±10%, H-Flip |
| Regularization | L2 + BatchNorm + Dropout(0.4) |

**Why CNN > ANN for images?**  
CNNs learn *local spatial features* (edges → textures → shapes) through **weight-shared convolutional filters**. This gives them:
- Far fewer parameters than flat fully-connected layers
- Translation invariance
- Hierarchical feature learning

In [ ]:
def build_cnn(filters=[32,64,128], kernel_size=(3,3), pool_size=(2,2),
              dropout_rate=0.4, dense_units=[256,128],
              learning_rate=1e-3, use_augmentation=True):
    """Build and compile a CNN with optional data augmentation."""
    inputs = layers.Input(shape=(28, 28, 1), name='input')
    x = inputs

    # ── Data Augmentation ──────────────────────────────────────────────────
    if use_augmentation:
        aug = models.Sequential([
            layers.RandomRotation(10/360),
            layers.RandomTranslation(0.1, 0.1),
            layers.RandomZoom(0.1),
            layers.RandomFlip('horizontal'),
        ], name='augmentation')
        x = aug(x)

    # ── Convolutional Blocks ─────────────────────────────────────────────
    for i, f in enumerate(filters):
        x = layers.Conv2D(f, kernel_size, padding='same',
                          kernel_regularizer=regularizers.l2(1e-4),
                          name=f'conv{i+1}_a')(x)
        x = layers.BatchNormalization(name=f'bn{i+1}_a')(x)
        x = layers.Activation('relu', name=f'relu{i+1}_a')(x)
        if i < len(filters) - 1:
            x = layers.Conv2D(f, kernel_size, padding='same',
                              kernel_regularizer=regularizers.l2(1e-4),
                              name=f'conv{i+1}_b')(x)
            x = layers.BatchNormalization(name=f'bn{i+1}_b')(x)
            x = layers.Activation('relu', name=f'relu{i+1}_b')(x)
        x = layers.MaxPooling2D(pool_size, name=f'pool{i+1}')(x)
        x = layers.Dropout(dropout_rate * 0.5, name=f'drop_conv{i+1}')(x)

    # ── FC Head ─────────────────────────────────────────────────────────
    x = layers.Flatten(name='flatten')(x)
    for j, units in enumerate(dense_units):
        x = layers.Dense(units, activation='relu',
                         kernel_regularizer=regularizers.l2(1e-4),
                         name=f'fc{j+1}')(x)
        x = layers.BatchNormalization(name=f'bn_fc{j+1}')(x)
        x = layers.Dropout(dropout_rate, name=f'drop_fc{j+1}')(x)

    outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='output')(x)
    model   = models.Model(inputs, outputs, name='CNN_FashionMNIST')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

cnn = build_cnn(use_augmentation=True)
cnn.summary()

In [ ]:
print('Training CNN...')
history_cnn = cnn.fit(
    X_train_4d, y_train,
    epochs=30,
    batch_size=64,
    validation_split=0.1,
    callbacks=get_callbacks('CNN'),
    verbose=1
)

In [ ]:
plot_curves(history_cnn, 'CNN')

In [ ]:
cnn_loss, cnn_acc = cnn.evaluate(X_test_4d, y_test, verbose=0)
print(f'CNN Test Accuracy : {cnn_acc*100:.2f}%')
print(f'CNN Test Loss     : {cnn_loss:.4f}')

---
## 📈 Step 5 — Evaluation: Confusion Matrix & Classification Report

In [ ]:
def plot_confusion_matrix(model, X, y, model_name):
    """Predict and plot a normalised seaborn confusion matrix."""
    y_pred = np.argmax(model.predict(X, verbose=0), axis=1)
    cm      = confusion_matrix(y, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    fig, ax = plt.subplots(figsize=(12, 10))
    fig.patch.set_facecolor('#1a1a2e')
    ax.set_facecolor('#16213e')

    sns.heatmap(cm_norm, annot=True, fmt='.2f',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                cmap='Blues', ax=ax, linewidths=0.5, linecolor='#333',
                annot_kws={'size': 8})

    ax.set_title(f'{model_name} — Confusion Matrix (Normalised)',
                 fontsize=13, fontweight='bold', pad=15, color='white')
    ax.set_xlabel('Predicted Label', labelpad=10, color='white')
    ax.set_ylabel('True Label',      labelpad=10, color='white')
    ax.tick_params(colors='white')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(f'../outputs/plots/nb_{model_name}_confusion.png',
                bbox_inches='tight', dpi=150, facecolor=fig.get_facecolor())
    plt.show()
    return y_pred

y_pred_ann = plot_confusion_matrix(ann, X_test_flat, y_test, 'ANN')

In [ ]:
print('Classification Report — ANN')
print('=' * 60)
print(classification_report(y_test, y_pred_ann, target_names=CLASS_NAMES, digits=4))

In [ ]:
y_pred_cnn = plot_confusion_matrix(cnn, X_test_4d, y_test, 'CNN')
print('Classification Report — CNN')
print('=' * 60)
print(classification_report(y_test, y_pred_cnn, target_names=CLASS_NAMES, digits=4))

---
## 🖼️ Step 6 — Prediction Visualisation

In [ ]:
def visualize_predictions(model, X_test, y_test, model_name, n_rows=4, n_cols=8):
    n = n_rows * n_cols
    idxs   = np.random.choice(len(X_test), n, replace=False)
    X_samp = X_test[idxs]
    y_true = y_test[idxs]
    y_pred = np.argmax(model.predict(X_samp, verbose=0), axis=1)

    fig = plt.figure(figsize=(n_cols * 1.8, n_rows * 2.2))
    fig.patch.set_facecolor('#1a1a2e')
    gs  = gridspec.GridSpec(n_rows, n_cols, figure=fig, hspace=0.6, wspace=0.3)
    fig.suptitle(f'{model_name} — Predictions  (🟢 correct | 🔴 wrong)',
                 fontsize=13, color='white', fontweight='bold', y=1.01)

    for i, (img, t, p) in enumerate(zip(X_samp, y_true, y_pred)):
        ax  = fig.add_subplot(gs[i // n_cols, i % n_cols])
        col = '#00e676' if t == p else '#ff5252'
        ax.imshow(img.squeeze(), cmap='gray')
        ax.set_title(f'T:{CLASS_NAMES[t][:5]}\nP:{CLASS_NAMES[p][:5]}',
                     fontsize=6, color=col, pad=2)
        ax.axis('off')
        for spine in ax.spines.values():
            spine.set_edgecolor(col); spine.set_linewidth(2)

    plt.tight_layout()
    plt.savefig(f'../outputs/plots/nb_{model_name}_predictions.png',
                bbox_inches='tight', dpi=150, facecolor=fig.get_facecolor())
    plt.show()

visualize_predictions(ann, X_test_flat, y_test, 'ANN')
visualize_predictions(cnn, X_test_4d,  y_test, 'CNN')

---
## 📊 Step 7 — Model Comparison

In [ ]:
results = {
    'Model'     : ['ANN', 'CNN'],
    'Accuracy %': [ann_acc*100, cnn_acc*100],
    'Loss'      : [ann_loss,    cnn_loss],
}
df = pd.DataFrame(results)
print(df.to_string(index=False))
print(f'\n🚀 CNN outperforms ANN by {(cnn_acc - ann_acc)*100:.2f} percentage points!')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.patch.set_facecolor('#1a1a2e')
colors = ['#00d2ff', '#a855f7']

for ax in (ax1, ax2):
    ax.set_facecolor('#16213e')
    ax.tick_params(colors='white')
    ax.yaxis.label.set_color('white')
    for s in ax.spines.values(): s.set_edgecolor('#444')

bars1 = ax1.bar(['ANN', 'CNN'], [ann_acc*100, cnn_acc*100], color=colors,
                edgecolor='white', lw=0.8, width=0.4)
ax1.set_title('Test Accuracy Comparison', color='white', fontsize=13, fontweight='bold')
ax1.set_ylabel('Accuracy (%)', color='white'); ax1.set_ylim(0, 100)
ax1.grid(axis='y', ls='--', alpha=0.4)
for bar, v in zip(bars1, [ann_acc*100, cnn_acc*100]):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{v:.2f}%', ha='center', color='white', fontweight='bold', fontsize=12)

bars2 = ax2.bar(['ANN', 'CNN'], [ann_loss, cnn_loss], color=colors,
                edgecolor='white', lw=0.8, width=0.4)
ax2.set_title('Test Loss Comparison', color='white', fontsize=13, fontweight='bold')
ax2.set_ylabel('Loss', color='white')
ax2.grid(axis='y', ls='--', alpha=0.4)
for bar, v in zip(bars2, [ann_loss, cnn_loss]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{v:.4f}', ha='center', color='white', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('../outputs/plots/nb_model_comparison.png',
            bbox_inches='tight', dpi=150, facecolor=fig.get_facecolor())
plt.show()

---
## 💾 Step 8 — Save Models

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

ann.save('../models/ANN_final.keras')
cnn.save('../models/CNN_final.keras')
print('✅ ANN saved → models/ANN_final.keras')
print('✅ CNN saved → models/CNN_final.keras')

---
## 💡 Step 9 — Real-World Mapping

| Application | How This Model Helps |
|-------------|---------------------|
| **E-commerce Auto-Tagging** | Automatically assign category labels (T-shirt, Dress, Sneaker…) to new product uploads — replaces manual tagging done by thousands of employees |
| **Visual Search** | "Shop the look" — given a photo, find similar products by comparing CNN feature embeddings |
| **Recommendation Systems** | Suggest items in the same predicted category as recently viewed products |
| **Inventory Management** | Auto-sort warehouse stock photos by garment type without human review |
| **Trend Analysis** | Track category-level upload frequency to detect fashion trends early |

### Deployment Path
```
User uploads image → Resize to 28×28 grayscale → Normalize → CNN inference → Return category + confidence
```
The saved `.keras` model can be served via **TensorFlow Serving**, **FastAPI**, or wrapped in a **Streamlit app** (see `app.py`).

---
## ✅ Summary

| Step | Status |
|------|--------|
| Dataset loaded & visualised | ✅ |
| Preprocessing (normalise, reshape) | ✅ |
| ANN built & trained | ✅ |
| CNN built & trained (with augmentation) | ✅ |
| Training curves plotted | ✅ |
| Confusion matrices plotted | ✅ |
| Classification reports printed | ✅ |
| Prediction grids visualised | ✅ |
| Models compared | ✅ |
| Models saved to disk | ✅ |
| Real-world applications explained | ✅ |